# wrap-forward-fn-generic — ex2: extend wrap_forward_fn with kwargs pass-through and is_differentiable

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wrap-forward-fn-generic`. Running the final beacon cell reports progress against the `Backprop: wrap forward fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: wrap forward fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wrap-forward-fn-generic`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wrap-forward-fn-generic"
DD_SUBTOPIC = "Backprop: wrap forward fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## wrap_forward_fn — quick refresher

`wrap_forward_fn` is the **factory** that turns a plain numerical fn (`torch.log`, `torch.multiply`, ...) into an autograd-aware version that (a) unboxes Tensor → raw, (b) runs the forward, (c) boxes the result back into a Tensor, and (d) attaches a Recipe so the reverse pass can replay the call.

```python
def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        out = Tensor(out_raw)
        # if any input is a tracked Tensor, attach a Recipe
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

The closure captures `fwd_fn` — one factory call replaces dozens of hand-written wrappers. Every wrapper shares the same unbox-call-box skeleton; only `fwd_fn` varies.

### Exercise 2 — extend wrap_forward_fn with kwargs pass-through and is_differentiable

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Create an extended wrap_forward_fn that threads kwargs through to both the forward call and the Recipe, and respects an is_differentiable flag to short-circuit Recipe construction for non-diff ops.
> Keywords: kwargs, is-differentiable, recipe, closure
> ```

**KCs targeted:** `wrap-forward-fn-generic`, `is-differentiable-flag`

Extend the simple `wrap_forward_fn` from the previous drill into the full ARENA version. The minimal `Tensor`, `Recipe`, and a `requires_grad` flag are scaffolded for you.

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)` so that the returned `tensor_func(*args, **kwargs)`:

1. **Unbox + call** as before — Tensors → `.array`, then    `fwd_fn(*raw_args, **kwargs)` (kwargs threaded through).
2. Compute `requires_grad = is_differentiable AND any(input is Tensor    with requires_grad=True)`. (No global toggle in this drill —    simpler than ARENA's three-conjunct version.)
3. **Box** the result: `out = Tensor(out_raw, requires_grad)`.
4. If `requires_grad` is True, **attach a Recipe**:    `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)` where    `parents = {i: a for i, a in enumerate(args) if isinstance(a, Tensor)}`.    Note `kwargs` lands in the Recipe verbatim — the reverse pass    needs them too (e.g. `dim=` for `sum_back`).
5. Otherwise leave `out.recipe = None`.

**The two new pieces** beyond drill 1:
- **kwargs pass-through to Recipe** — ARENA's `sum_back` needs to   know the `dim=` and `keepdim=` that the forward `sum` was called   with; the Recipe carries them.
- **`is_differentiable=False`** — for ops like `torch.eq` that return   bools, we wrap them so they accept Tensors, but we skip Recipe   construction (no backward pass possible).

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict


class Tensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
        self.requires_grad = requires_grad
        self.recipe = None
    def __repr__(self):
        return f'Tensor({self.array.tolist()}, requires_grad={self.requires_grad})'


def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    """Tensor-aware wrapper with Recipe attachment and is_differentiable gating."""
    raise NotImplementedError()


def _test_ex2():
    # --- baseline: wrap torch.log, requires_grad propagates ---
    tlog = wrap_forward_fn(t.log)
    a = Tensor(t.tensor([1.0, t.e]), requires_grad=True)
    b = tlog(a)
    assert b.requires_grad, 'requires_grad must propagate from input'
    assert b.recipe is not None, 'Recipe must be attached when requires_grad=True'
    assert b.recipe.func is t.log
    assert b.recipe.parents == {0: a}, f'parents wrong: {b.recipe.parents}'
    assert b.recipe.kwargs == {}, f'kwargs wrong: {b.recipe.kwargs}'
    assert t.allclose(b.array, t.tensor([0.0, 1.0]), atol=1e-5)

    # --- no requires_grad anywhere → no Recipe, no requires_grad ---
    a_off = Tensor(t.tensor([1.0, t.e]), requires_grad=False)
    b_off = tlog(a_off)
    assert b_off.requires_grad is False, 'no input wants grad → output mustnt either'
    assert b_off.recipe is None, 'no Recipe when requires_grad=False'

    # --- binary op with one tracked input + one raw scalar ---
    tmul = wrap_forward_fn(t.multiply)
    x = Tensor(t.tensor([2.0, 3.0]), requires_grad=True)
    z = tmul(x, 5.0)
    assert z.requires_grad
    assert z.recipe.parents == {0: x}, f'scalar must NOT be in parents: {z.recipe.parents}'
    assert z.recipe.args[1] == 5.0, 'raw scalar passes through to recipe.args'
    assert t.allclose(z.array, t.tensor([10.0, 15.0]))

    # --- kwargs threaded through to BOTH forward call and Recipe ---
    tsum = wrap_forward_fn(t.sum)
    m = Tensor(t.tensor([[1.0, 2.0], [3.0, 4.0]]), requires_grad=True)
    s = tsum(m, dim=1, keepdim=True)
    assert t.allclose(s.array, t.tensor([[3.0], [7.0]])), f'kwargs not used in fwd: {s.array}'
    assert s.recipe.kwargs == {'dim': 1, 'keepdim': True}, (
        f'kwargs missing from Recipe: {s.recipe.kwargs}'
    )

    # --- is_differentiable=False short-circuits Recipe ---
    teq = wrap_forward_fn(t.eq, is_differentiable=False)
    p = Tensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    q = Tensor(t.tensor([1.0, 5.0, 3.0]), requires_grad=True)
    r = teq(p, q)
    assert r.requires_grad is False, 'is_differentiable=False forces requires_grad=False'
    assert r.recipe is None, 'is_differentiable=False forces no Recipe'
    assert t.equal(r.array, t.tensor([True, False, True])), f'eq value wrong: {r.array}'

    # --- two tracked inputs → parents has both ---
    x2 = Tensor(t.tensor([2.0]), requires_grad=True)
    y2 = Tensor(t.tensor([3.0]), requires_grad=True)
    z2 = tmul(x2, y2)
    assert z2.recipe.parents == {0: x2, 1: y2}, f'two-input parents: {z2.recipe.parents}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, field


@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict


class Tensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
        self.requires_grad = requires_grad
        self.recipe = None
    def __repr__(self):
        return f'Tensor({self.array.tolist()}, requires_grad={self.requires_grad})'


def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        # 1. Unbox.
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        # 2. Call (kwargs thread through).
        out_raw = fwd_fn(*raw_args, **kwargs)
        # 3. Compute requires_grad — both gates.
        requires_grad = is_differentiable and any(
            isinstance(a, Tensor) and a.requires_grad for a in args
        )
        # 4. Box.
        out = Tensor(out_raw, requires_grad)
        # 5. Attach Recipe if tracked. parents = dict-by-argidx filtered to Tensors.
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, Tensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

**Why `parents` filters non-Tensors.** A `Tensor * 5.0` call has `args = (Tensor, 5.0)`. We can't store `5.0` in `parents` because the reverse pass iterates `parents.items()` and dispatches a back fn per parent — there's no gradient to compute for a raw scalar. The dict-by-argidx-with-Tensor-filter pattern keeps the keys aligned with the original arg positions (so `BACK_FUNCS.get_back_func(fn, argnum)` uses the right argnum).

**Why kwargs land verbatim in Recipe.** `sum_back(grad_out, out, x, dim=..., keepdim=...)` needs the same `dim`/`keepdim` the forward used, or the broadcast-shape inverse comes out wrong. ARENA's reverse pass does `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)` — kwargs flow back through, same shape.

**Why `is_differentiable` short-circuits before the Recipe.** `torch.eq` returns bools — there is no meaningful gradient. We still want to wrap it so users can write `Tensor(...) == Tensor(...)`, but `requires_grad` must be False on the output (and no Recipe → no wasted memory).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()